<a href="https://colab.research.google.com/github/longlive13/Optimizing-Marketing-ROI-through-Customer-Segmentation-and-Channel-Strategy/blob/main/marketingAnalysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

# 1. 데이터 로드
file_path = "/content/drive/MyDrive/marketing_roi/bank-full.csv"
df = pd.read_csv(file_path, sep=';')


class MarketingSystem:
    def __init__(self, df: pd.DataFrame):
        self.df = df.copy()

        # 분석용 가정값
        self.cost_map = {
            "Email": 0.5,
            "SMS": 1.2,
            "Phone": 5.0
        }
        self.conversion_value = 50.0  # 가입 1건당 기대 수익(가정)

    def validate_raw_data(self):
        required_cols = [
            "age", "job", "balance", "contact", "campaign", "y"
        ]
        missing = [c for c in required_cols if c not in self.df.columns]
        if missing:
            raise ValueError(f"필수 컬럼 누락: {missing}")

        assert self.df["age"].notna().all(), "age null 존재"
        assert self.df["campaign"].notna().all(), "campaign null 존재"
        assert (self.df["campaign"] > 0).all(), "campaign은 1 이상이어야 함"
        assert self.df["y"].isin(["yes", "no"]).all(), "y 값 이상"
        return self

    def create_segments(self):
        # 연령 세그먼트
        self.df["age_group"] = pd.cut(
            self.df["age"],
            bins=[0, 29, 49, 120],
            labels=["young", "middle", "senior"]
        )

        # 잔고 세그먼트
        self.df["balance_segment"] = pd.cut(
            self.df["balance"],
            bins=[-np.inf, 999, 4999, np.inf],
            labels=["low", "mid", "high"]
        )
        return self

    def assign_channel(self, random_state: int = 42):
        """
        원본 contact를 참고해서 분석용 channel 생성
        - contact == telephone 이면 Phone 비중 높임
        - contact == cellular 이면 SMS/Phone 혼합
        - 젊고 잔고 낮으면 Email 비중 확대
        - senior/high balance/management는 Phone 비중 확대
        """
        rng = np.random.default_rng(random_state)

        def choose_channel(row):
            age = row["age"]
            balance = row["balance"]
            job = row["job"]
            contact = row["contact"]

            # 기본 확률
            probs = {"Email": 0.2, "SMS": 0.4, "Phone": 0.4}

            # 원본 contact 반영
            if contact == "telephone":
                probs = {"Email": 0.05, "SMS": 0.15, "Phone": 0.80}
            elif contact == "cellular":
                probs = {"Email": 0.20, "SMS": 0.55, "Phone": 0.25}
            else:
                probs = {"Email": 0.30, "SMS": 0.35, "Phone": 0.35}

            # 젊고 저잔고 → 저비용 채널 선호
            if age < 35 and balance < 1000:
                probs["Email"] += 0.25
                probs["Phone"] -= 0.15
                probs["SMS"] -= 0.10

            # 고연령 / 관리직 / 고잔고 → Phone 선호
            if age >= 50 or job == "management" or balance >= 5000:
                probs["Phone"] += 0.20
                probs["Email"] -= 0.10
                probs["SMS"] -= 0.10

            # 음수 방지
            probs = {k: max(v, 0.0) for k, v in probs.items()}

            # 합 1로 정규화
            total = sum(probs.values())
            probs = {k: v / total for k, v in probs.items()}

            channels = list(probs.keys())
            pvals = list(probs.values())
            return rng.choice(channels, p=pvals)

        self.df["channel"] = self.df.apply(choose_channel, axis=1)

        assert self.df["channel"].notna().all(), "channel null 존재"
        assert self.df["channel"].isin(["Email", "SMS", "Phone"]).all(), "channel 값 이상"
        return self

    def create_metrics(self):
        # 분석용 전환 여부 컬럼
        self.df["converted"] = self.df["y"].map({"yes": 1, "no": 0})

        # 비용 = 접촉 횟수 * 채널 단가
        self.df["cost"] = self.df["campaign"] * self.df["channel"].map(self.cost_map)

        # 수익 = 전환 시 기대 수익
        self.df["revenue"] = self.df["converted"] * self.conversion_value

        # 연락 1회당 기대 수익
        self.df["revenue_per_contact"] = self.df["revenue"] / self.df["campaign"]

        # ROI
        self.df["roi"] = (self.df["revenue"] - self.df["cost"]) / self.df["cost"]

        assert (self.df["cost"] > 0).all(), "cost는 0보다 커야 함"
        assert (self.df["revenue"] >= 0).all(), "revenue 음수 존재"
        return self

    def get_channel_report(self):
        report = (
            self.df.groupby("channel")
            .agg(
                customers=("y", "count"),
                conversions=("converted", "sum"),
                conversion_rate=("converted", "mean"),
                total_cost=("cost", "sum"),
                total_revenue=("revenue", "sum"),
                avg_roi=("roi", "mean"),
                revenue_per_contact=("revenue_per_contact", "mean")
            )
            .reset_index()
        )
        report["conversion_rate"] = report["conversion_rate"] * 100
        return report

    def get_segment_report(self):
        report = (
            self.df.groupby(["channel", "job", "age_group", "balance_segment"])
            .agg(
                customers=("y", "count"),
                conversions=("converted", "sum"),
                conversion_rate=("converted", "mean"),
                avg_roi=("roi", "mean"),
                total_cost=("cost", "sum"),
                total_revenue=("revenue", "sum")
            )
            .reset_index()
        )
        report["conversion_rate"] = report["conversion_rate"] * 100
        return report

    def get_contact_frequency_report(self):
        def campaign_band(x):
            if x <= 2:
                return "1-2"
            elif x <= 4:
                return "3-4"
            elif x <= 6:
                return "5-6"
            else:
                return "7+"

        self.df["campaign_band"] = self.df["campaign"].apply(campaign_band)

        report = (
            self.df.groupby(["campaign_band", "channel"])
            .agg(
                customers=("y", "count"),
                conversion_rate=("converted", "mean"),
                avg_roi=("roi", "mean"),
                avg_cost=("cost", "mean")
            )
            .reset_index()
        )
        report["conversion_rate"] = report["conversion_rate"] * 100
        return report


# 실행
system = (
    MarketingSystem(df)
    .validate_raw_data()
    .create_segments()
    .assign_channel(random_state=42)
    .create_metrics()
)

final_df = system.df
channel_report = system.get_channel_report()
segment_report = system.get_segment_report()
contact_report = system.get_contact_frequency_report()

print("=== Channel Report ===")
print(channel_report.head())

print("\n=== Segment Report ===")
print(segment_report.head())

print("\n=== Contact Frequency Report ===")
print(contact_report.head())

#  저장
final_df.to_csv("/content/drive/MyDrive/marketing_roi/bank_processed.csv", index=False)

/tmp/ipykernel_820/4047076726.py:146: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  self.df.groupby(["channel", "job", "age_group", "balance_segment"])


=== Channel Report ===
  channel  customers  conversions  conversion_rate  total_cost  total_revenue  \
0   Email      10791         1105        10.240015     14623.5        55250.0   
1   Phone      16375         1958        11.957252    230295.0        97900.0   
2     SMS      18045         2226        12.335827     59580.0       111300.0   

    avg_roi  revenue_per_contact  
0  6.065624             3.532812  
1 -0.198100             4.009499  
2  2.538038             4.245645  

=== Segment Report ===
  channel     job age_group balance_segment  customers  conversions  \
0   Email  admin.     young             low        296           35   
1   Email  admin.     young             mid         31            4   
2   Email  admin.     young            high          1            1   
3   Email  admin.    middle             low        779           84   
4   Email  admin.    middle             mid        192           21   

   conversion_rate    avg_roi  total_cost  total_revenue  
0 